In [1]:
import paddle
from paddle.io import DataLoader, random_split
from paddle.vision.datasets import Cifar10
from darknet import Darknet19

/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [2]:
transform = paddle.vision.transforms.Compose([
    paddle.vision.transforms.ToTensor(),
    paddle.vision.transforms.Resize((224, 224)),
    paddle.vision.transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])

mnist = Cifar10(mode='train', backend='cv2', transform=transform)
mnist_test = Cifar10(mode='test', backend='cv2', transform=transform)

mnist_train, mnist_val = random_split(mnist, [round(0.8*len(mnist)), round(0.2*len(mnist))])

W0510 11:49:57.834501 145041 gpu_resources.cc:116] Please NOTE: device: 0, GPU Compute Capability: 7.0, Driver API Version: 12.0, Runtime API Version: 11.8


In [10]:
train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
val_loader = DataLoader(mnist_val, batch_size=128, shuffle=False)
test_loader = DataLoader(mnist_test, batch_size=128, shuffle=False)

# plt.imshow(list(train_loader)[0][0][4].numpy().reshape([28, 28]))

img, label = next(iter(train_loader))
img.shape, label.shape

(paddle.Size([128, 3, 224, 224]), paddle.Size([128]))

In [11]:
model = Darknet19()
paddle.summary(model, (1, 3, 224, 224))


-------------------------------------------------------------------------------
   Layer (type)         Input Shape          Output Shape         Param #    
     Conv2D-39       [[1, 3, 224, 224]]   [1, 32, 224, 224]         864      
  BatchNorm2D-37    [[1, 32, 224, 224]]   [1, 32, 224, 224]         128      
   LeakyReLU-37     [[1, 32, 224, 224]]   [1, 32, 224, 224]          0       
 BasicConvLayer-37   [[1, 3, 224, 224]]   [1, 32, 224, 224]          0       
   MaxPool2D-11     [[1, 32, 224, 224]]   [1, 32, 112, 112]          0       
     Conv2D-40      [[1, 32, 112, 112]]   [1, 64, 112, 112]       18,432     
  BatchNorm2D-38    [[1, 64, 112, 112]]   [1, 64, 112, 112]         256      
   LeakyReLU-38     [[1, 64, 112, 112]]   [1, 64, 112, 112]          0       
 BasicConvLayer-38  [[1, 32, 112, 112]]   [1, 64, 112, 112]          0       
   MaxPool2D-12     [[1, 64, 112, 112]]    [1, 64, 56, 56]           0       
     Conv2D-41       [[1, 64, 56, 56]]     [1, 128, 56, 56]   

{'total_params': 19842026, 'trainable_params': 19827626}

In [12]:
optimizer = paddle.optimizer.AdamW(learning_rate=0.01, parameters=model.parameters(), weight_decay=0.001)
loss_fn = paddle.nn.CrossEntropyLoss()

In [13]:
model.train()
for epoch in range(5):
    for batch_id, (images, labels) in enumerate(train_loader):        
        logits = model(images)               # [64, 100, 1, 1]
        logits = logits.squeeze()       
        loss = loss_fn(logits, labels)        
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        if batch_id % 30 == 0:
            print(f'Epoch: {epoch}, Batch_id: {batch_id}, Loss: {loss.item()}')
    model.eval()
    correct, total = 0, 0
    with paddle.no_grad():
        for images, labels in val_loader:
            logits = model(images)
            preds = paddle.argmax(logits, axis=1)
            correct += (preds == labels).numpy().sum()
            total += labels.shape[0]    
    acc = correct / total
    print(f"Epoch {epoch} Finished, Test Acc: {acc:.4f}")
    model.train()

/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle/nn/layer/norm.py:885: UserWarning: When training, we now always track global mean and variance.
  warnings.warn(


Epoch: 0, Batch_id: 0, Loss: 2.537771463394165
Epoch: 0, Batch_id: 30, Loss: 2.318934917449951
Epoch: 0, Batch_id: 60, Loss: 2.1503095626831055
Epoch: 0, Batch_id: 90, Loss: 2.402278184890747
Epoch: 0, Batch_id: 120, Loss: 2.4605178833007812
Epoch: 0, Batch_id: 150, Loss: 2.064668893814087
Epoch: 0, Batch_id: 180, Loss: 2.0085151195526123
Epoch: 0, Batch_id: 210, Loss: 1.9286985397338867
Epoch: 0, Batch_id: 240, Loss: 1.7405458688735962
Epoch: 0, Batch_id: 270, Loss: 1.907333493232727
Epoch: 0, Batch_id: 300, Loss: 1.890590786933899
Epoch 0 Finished, Test Acc: 0.3367
Epoch: 1, Batch_id: 0, Loss: 1.7975025177001953
Epoch: 1, Batch_id: 30, Loss: 1.7660013437271118
Epoch: 1, Batch_id: 60, Loss: 1.8041303157806396
Epoch: 1, Batch_id: 90, Loss: 1.661714792251587
Epoch: 1, Batch_id: 120, Loss: 1.7105209827423096
Epoch: 1, Batch_id: 150, Loss: 1.5879127979278564
Epoch: 1, Batch_id: 180, Loss: 1.590848684310913
Epoch: 1, Batch_id: 210, Loss: 1.5671000480651855
Epoch: 1, Batch_id: 240, Loss: 1.

In [15]:
from utils import save_weight

save_weight(model, './darknet')